# Exercise Class 5: Social Loss, Taylor-Rule Stabilisation, and Optimal Policy
## Student Starter Notebook

**Macroeconomics B -- Chapters 20-22**

By the end of this notebook you will be able to:
1. Plot iso-loss contours and identify which feasible points have lower social cost.
2. Compute AS-AD equilibria for demand and supply shocks under different Taylor-rule parameters.
3. Derive and visualise the optimal policy point, policy line, and loss landscape after a supply shock.
4. Simulate and compare inflation paths under static expectations and credible rational expectations.

**Table of contents**<a id='toc0_'></a>
- 1. [Parameters and helper functions](#toc1_)
- 2. [Q1(e): Iso-loss contours](#toc2_)
- 3. [Q2(e): Taylor-rule AD -- demand vs. supply shock table and trade-off curve](#toc3_)
- 4. [Q3(f): Optimal policy -- loss landscape and sensitivity analysis](#toc4_)
- 5. [Q4(d): Rational expectations -- static vs. credible-RE disinflation](#toc5_)
- 6. [Summary](#toc6_)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Global plot settings (matches Programming for Economists style)
plt.rcParams.update({'axes.grid': True, 'grid.color': 'black',
                     'grid.alpha': 0.25, 'grid.linestyle': '--'})
plt.rcParams.update({'font.size': 14})

# Reproducibility
rng = np.random.default_rng(2026)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']


## 1. <a id='toc1_'></a>[Parameters and helper functions](#toc0_)

*Corresponds to slides: "Model Blocks" (Lecture 7, frame 19) and "The Optimal Policy Problem" (Lecture 7, frame 32).*

All exercises share the same baseline parameter dictionary and three reusable functions.
Parameters can be modified at the top of each section for comparative statics.


In [ ]:
# -------------------------------------------------------
# Baseline parameters
# -------------------------------------------------------
par = {
    'gamma':   0.50,   # AS slope (price pressure per unit of output gap)
    'kappa':   2.00,   # inflation weight in social loss function
    'alpha2':  1.20,   # IS sensitivity to the real interest rate
    'h':       1.00,   # Taylor-rule response to the inflation gap
    'b':       0.50,   # Taylor-rule response to the output gap
    'pi_star': 2.00,   # inflation target (%)
}

a_baseline = par['alpha2'] * par['h'] / (1.0 + par['alpha2'] * par['b'])
a_optimal  = par['gamma'] * par['kappa']
h_optimal  = a_optimal / (par['alpha2'] / (1.0 + par['alpha2'] * par['b']))
print(f'Baseline AD slope  a = {a_baseline:.4f}')
print(f'Optimal  AD slope  a* = gamma*kappa = {a_optimal:.4f}')
print(f'Taylor h needed for optimum (b=0.5): h* = {h_optimal:.4f}')


In [ ]:
def ad_slope(p):
    # Return AD slope: a = alpha2 * h / (1 + alpha2 * b)
    return p['alpha2'] * p['h'] / (1.0 + p['alpha2'] * p['b'])


def equilibrium(p, z, s):
    # Solve AS-AD system for output gap and inflation gap.
    # AS:  pi_hat = gamma * y_hat + s  (with pi_e = pi_star)
    # AD:  y_hat  = z - a * pi_hat
    #
    # Args:
    #   p (dict): parameters with 'gamma', 'alpha2', 'h', 'b'
    #   z (float): demand-side shift
    #   s (float): supply shock
    # Returns: (y_hat, pi_hat)

    # a. AD slope
    a = ad_slope(p)

    # b. solve: pi_hat*(1 + gamma*a) = gamma*z + s
    pi_hat = (p['gamma'] * z + s) / (1.0 + p['gamma'] * a)

    # c. output gap from AD
    y_hat = z - a * pi_hat

    return y_hat, pi_hat


def optimal_response(p, s):
    # Compute one-period optimal gaps.
    # Solves: min_{y_hat}  y_hat^2 + kappa*(gamma*y_hat + s)^2
    # Solution:
    #   y_hat*  = -gamma*kappa*s / (1 + gamma^2*kappa)
    #   pi_hat* =  s / (1 + gamma^2*kappa)
    #
    # Args:
    #   p (dict): parameters with 'gamma', 'kappa'
    #   s (float): supply shock
    # Returns: (y_star, pi_star)

    # a. unpack
    gamma = p['gamma']
    kappa = p['kappa']

    # b. common denominator
    denom = 1.0 + gamma**2 * kappa

    # c. optimal gaps
    y_star  = -gamma * kappa * s / denom
    pi_star = s / denom

    return y_star, pi_star


## 2. <a id='toc2_'></a>[Q1(e): Iso-loss contours](#toc0_)

*Corresponds to slides: "The Period Social Loss Function" and "Iso-Loss Curves I--II" (Lecture 7, frames 10--12).*

The social loss $SL = \hat{y}^2 + \kappa\hat{\pi}^2 = v$ defines an ellipse in $(\hat{y}, \hat{\pi})$ space.
The $\hat{y}$-axis intercepts $\pm\sqrt{v}$ are independent of $\kappa$, while the $\hat{\pi}$-axis intercepts $\pm\sqrt{v/\kappa}$ shrink as $\kappa$ rises.
A higher $\kappa$ compresses contours in the inflation direction: the same output gap is tolerated but a given inflation deviation costs more.

**Your task:**
1. Define a meshgrid over $\hat{y}$ and $\hat{\pi}$ ranging from $-3.5$ to $3.5$.
2. Draw iso-loss contours for `kappa_A = 0.5` (solid) and `kappa_B = 2.0` (dashed) at levels `[0.5, 1.0, 2.0, 4.0, 7.0]`.
3. Mark points $A=(-1, 1)$ and $B=(0.5, -1.5)$. Annotate each with its social loss for $\kappa=2$.
4. Print the losses for both $\kappa$ values and state which point wins under each.


In [ ]:
# -------------------------------------------------------
# Parameters -- edit to explore
# -------------------------------------------------------
kappa_A = 0.5   # kappa for first set of contours   <-- edit to explore
kappa_B = 2.0   # kappa for second set of contours  <-- edit to explore
# -------------------------------------------------------

v_levels = [0.5, 1.0, 2.0, 4.0, 7.0]

# write your code here


**What is done:** Iso-loss contours are plotted for two values of $\kappa$. Points $A$ and $B$ are marked and their losses computed numerically.

**Why it is useful:** The contour map is the graphical foundation of the optimal-policy derivation in Q3. The authority wants the lowest contour that is still feasible given the AS constraint. Understanding the shape -- and how $\kappa$ changes it -- is an **exam core skill**.

**Task:** Change `kappa_A` to `4.0`. Describe how the ellipses change and recompute the loss ranking. Which point is preferred under a very inflation-focused central bank?


In [ ]:
# write your code here


## 3. <a id='toc3_'></a>[Q2(e): Taylor-rule AD -- demand vs. supply shock table and trade-off curve](#toc0_)

*Corresponds to slides: "Demand Shocks: Stabilisation Without a Trade-Off" and "Supply Shocks: Why the Trade-Off Is Real" (Lecture 7, frames 22--25).*

The AS-AD equilibrium is:
$$\hat{\pi} = \frac{\gamma z + s}{1+\gamma a}, \qquad \hat{y} = \frac{z - as}{1+\gamma a}.$$
For a **demand shock**: $z=v>0$, $s=0$. For a **supply shock**: $z=0$, $s>0$.
We vary $h \in \{0, 0.5, 1, 2\}$ with $b=0.5$ fixed.

**Your task:**
1. Loop over `h_values`, call `equilibrium()` for both a demand shock (`z=2`, `s=0`) and a supply shock (`z=0`, `s=2`), and collect results into a `pd.DataFrame`.
2. Print the table.
3. Using a fine grid of $h$ values, trace the trade-off curve in $(\hat{y}, \hat{\pi})$ space for the supply shock. Mark the four discrete points and the two limiting points at $h=0$ and $h\to\infty$.


In [ ]:
# -------------------------------------------------------
# Parameters -- edit to explore
# -------------------------------------------------------
z_demand = 2.0
s_supply = 2.0
h_values = [0.0, 0.5, 1.0, 2.0]
b_fixed  = 0.5
# -------------------------------------------------------

# write your code here


In [ ]:
# -------------------------------------------------------
# Supply-shock trade-off curve: trace (y_hat, pi_hat) as h varies
# -------------------------------------------------------
h_fine   = np.linspace(0.0, 5.0, 300)

# write your code here


**What is done:** A table and a continuous trade-off curve show equilibrium gaps for demand and supply shocks as $h$ varies from 0 to 5.

**Why it is useful:** The table makes the central lesson of Chapter 20 concrete: **demand shocks can be stabilised without a trade-off** (both gaps fall as $h$ rises), while **supply shocks impose a trade-off** (lower inflation requires accepting a more negative output gap). This distinction is the most important exam idea from Lecture 7.

**Task:** Change `s_supply` to `3.0` and `z_demand` to `3.0`. Does the trade-off become more or less severe? What happens to the two endpoints of the curve?


In [ ]:
# write your code here


## 4. <a id='toc4_'></a>[Q3(f): Optimal policy -- loss landscape and sensitivity analysis](#toc0_)

*Corresponds to slides: "The Optimal Policy Problem", "Optimal Responses to an Adverse Supply Shock", and "What Changes with kappa and gamma?" (Lecture 7, frames 32--36).*

The central bank minimises $\hat{y}^2 + \kappa(\gamma\hat{y}+s)^2$ subject to AS.
The FOC yields the **policy line** $\hat{y} = -\gamma\kappa\hat{\pi}$.
The optimal gaps are:
$$\hat{y}^* = -\frac{\gamma\kappa}{1+\gamma^2\kappa}s, \qquad \hat{\pi}^* = \frac{s}{1+\gamma^2\kappa}.$$

**Your task:**
1. Compute the optimal point using `optimal_response(par, s_shock)`.
2. Plot iso-loss contours, the shifted AS curve ($\hat{\pi}=\gamma\hat{y}+s$), the policy line ($\hat{y}=-\gamma\kappa\hat{\pi}$), and mark the optimal point with a star.
3. Build a sensitivity table varying $\kappa \in \{0.25, 0.5, 1, 2, 4, 8\}$: report $\hat{y}^*$, $\hat{\pi}^*$, $a^*=\gamma\kappa$, and $h^*$.
4. Plot both optimal gaps as functions of $\kappa$ on a fine grid.


In [ ]:
# -------------------------------------------------------
# Parameters -- edit to explore
# -------------------------------------------------------
s_shock = 2.0   # supply shock  <-- edit to explore
kappa   = par['kappa']
gamma   = par['gamma']
# -------------------------------------------------------

# write your code here


In [ ]:
# -------------------------------------------------------
# Sensitivity: vary kappa with gamma=0.5, s=2 fixed
# -------------------------------------------------------
kappa_range = np.array([0.25, 0.50, 1.00, 2.00, 4.00, 8.00])
denom_h = par['alpha2'] / (1.0 + par['alpha2'] * par['b'])  # alpha2/(1+alpha2*b)

# write your code here


**What is done:** The loss landscape (iso-loss contours), shifted AS curve, and optimal policy line are plotted together. The optimal point -- where AS is tangent to the lowest reachable contour -- is marked. A sensitivity table and figure show how $\hat{y}^*$ and $\hat{\pi}^*$ vary with $\kappa$.

**Why it is useful:** The figure makes the **tangency argument** visible: the target $(0,0)$ is off the shifted AS curve and therefore infeasible. The sensitivity analysis connects $\kappa$ to required Taylor-rule parameters -- an **exam-relevant calculation**.

**Task:** Change `s_shock` to `1.5`. Verify that both optimal gaps scale proportionally with $s$. Does the policy line change?


In [ ]:
# write your code here


## 5. <a id='toc5_'></a>[Q4(d): Rational expectations -- static vs. credible-RE disinflation](#toc0_)

*Corresponds to slides: "A Credible Disinflation: Static Expectations versus Forward Looking" (Lecture 8, frame 8) and "What Rational Expectations Do to Persistence" (Lecture 8, frame 19).*

The central bank credibly announces a target reduction from $\pi_0 = 10\%$ to $\pi^* = 2\%$.

Under **static expectations** ($\pi^e_t = \pi_{t-1}$), combining AS and AD gives:
$$\hat{\pi}_t = \frac{\hat{\pi}_{t-1}}{1+\gamma a}, \quad\hat{y}_t = -a\hat{\pi}_t.$$
Inflation decays at rate $\beta = 1/(1+\gamma a)$ and the bank must maintain a negative output gap throughout.

Under **credible RE**, agents immediately set $\pi^e_t = \pi^*$ from $t=1$: inflation jumps to target at zero output cost.

**Your task:**
1. Compute the AD slope `a_base` and damping factor `beta`.
2. Preallocate arrays with `np.empty` and simulate both paths over $T=20$ periods.
3. Plot three panels: (a) inflation paths, (b) output gap under static expectations, (c) forecast error under static expectations as a bar chart.
4. Compute and compare the social loss at $t=3$ under both regimes.


In [ ]:
# -------------------------------------------------------
# Parameters -- edit to explore
# -------------------------------------------------------
T          = 20      # horizon (periods)  <-- edit to explore
pi_initial = 10.0    # initial inflation (%)  <-- edit to explore
pi_target  = 2.0     # new inflation target (%)
# -------------------------------------------------------

pi0_hat = pi_initial - pi_target   # initial inflation gap

# write your code here


In [ ]:
# write your code here


**What is done:** The model is simulated for $T=20$ periods after a credible target announcement ($\pi_0=10\% \to \pi^*=2\%$). Three panels show (a) inflation paths, (b) output gap under static expectations, and (c) forecast errors under static expectations.

**Why it is useful:** The simulation makes the **credibility dividend** concrete: a credible announcement eliminates the output cost of disinflation entirely. Under static expectations, slow convergence drives a persistent recession -- this motivated central-bank independence and inflation targeting.

**Task:** Change `pi_initial` to `6.0` (closer to the euro-area 2022 episode). How many periods does it take for static-expectations inflation to fall within 0.5 percentage points of target?


In [ ]:
# write your code here


## 6. <a id='toc6_'></a>[Summary](#toc0_)

| Equation | Meaning |
|---|---|
| $SL_t = \hat{y}_t^2 + \kappa\hat{\pi}_t^2$ | **Social loss**: quadratic penalties on output and inflation gaps |
| $d\hat{\pi}/d\hat{y}\big|_{SL} = -\hat{y}/(\kappa\hat{\pi})$ | **Iso-loss slope**: tangency condition |
| $a = \alpha_2 h / (1 + \alpha_2 b)$ | **AD slope**: $h$ creates the slope, $b$ reduces the shift |
| $\hat{y}_t = z_t - a\hat{\pi}_t$ | **AD curve** from IS + Taylor rule |
| $\hat{y}^* = -\gamma\kappa s / (1+\gamma^2\kappa)$ | **Optimal output gap** after supply shock |
| $\hat{\pi}^* = s / (1 + \gamma^2\kappa)$ | **Optimal inflation gap** after supply shock |
| $a = \gamma\kappa$ | **Taylor rule implements optimum** when AD slope equals this |
| $\hat{y}_t = v_t - \alpha_2\hat{\rho}_t$ | **Policy Ineffectiveness Proposition** (old-info RE) |
| $\hat{\pi}_t = \hat{\pi}_{t-1}/(1+\gamma a)$ | **Static-expectations decay** during disinflation |

**Socrative room:** MACROECONOMICSB
